In [ ]:
# ==============================================================================
# NOTEBOOK: 00_nrt_pipeline_and_audit
# DESCRIPCIÓN: Pipeline NRT (Ingesta + RAW + Bronze) + Inventario + Auditoría
# ==============================================================================
from pyspark.sql import functions as F

print("================================================================================")
print("🚀 INICIANDO CANALIZACIÓN DE DATOS NRT, INVENTARIO Y AUDITORÍA DE DATOS")
print("================================================================================\n")

# 1. FASE DE INGESTA LANDING
print("--- 1. FASE DE INGESTA LANDING ---")
ingesters = [
    ("DGT Traffic", "01_landing_dgt_traffic"),
    ("Weather", "01_landing_weather"),
    ("NASA FIRMS", "01_landing_nasa_nrt")
]

for name, nb in ingesters:
    try:
        res = mssparkutils.notebook.run(nb)
        print(f"✅ Ingesta {name} completada. Salida: {res}")
    except Exception as e:
        print(f"❌ Error en ingesta {name}: {e}")

# 2. FASE PROCESAMIENTO RAW
print("\n--- 2. FASE PROCESAMIENTO RAW ---")
try:
    mssparkutils.notebook.run("02_raw_realtime")
    print("✅ Archivos RAW actualizados en Parquet.")
except Exception as e:
    print(f"❌ Error en la capa RAW: {e}")

# 3. FASE PROMOCIÓN A BRONZE
print("\n--- 3. FASE PROMOCIÓN A BRONZE ---")
try:
    mssparkutils.notebook.run("03_bronze_realtime")
    print("✅ Tablas Delta Bronze actualizadas.")
except Exception as e:
    print(f"❌ Error en la capa Bronze: {e}")

# 4. INVENTARIO Y LISTADO DE ARCHIVOS
print("\n" + "="*80)
print("📦 INVENTARIO Y CONTEO DETALLADO DE ARCHIVOS REALTIME")
print("="*80)

def list_and_count_files(path: str, max_list: int = 5):
    file_list = []
    try:
        items = mssparkutils.fs.ls(path)
        for item in items:
            if item.isDir:
                sub_count, sub_list = list_and_count_files(item.path, max_list)
                file_list.extend(sub_list)
            elif not item.name.startswith("_") and not item.name.startswith("."):
                file_list.append(item.path)
    except Exception:
        pass
    return len(file_list), file_list

landing_dirs = [
    ("DGT Traffic (XML)", "Files/landing/realtime/dgt_traffic"),
    ("Weather (JSON)", "Files/landing/realtime/weather"),
    ("NASA FIRMS (CSV)", "Files/landing/realtime/nasa_nrt")
]

print("\n📂 LANDING REALTIME:")
for name, path in landing_dirs:
    total_f, files = list_and_count_files(path)
    print(f"\n   • {name} [{path}]: Total = {total_f:,} archivos.")
    if files:
        for f_path in files[:5]:
            print(f"       - {f_path}")
        if total_f > 5:
            print(f"       ... y {total_f - 5:,} archivos más.")

raw_dirs = [
    ("DGT Traffic RAW", "Files/raw/realtime/dgt_traffic"),
    ("Weather RAW", "Files/raw/realtime/weather"),
    ("NASA FIRMS RAW", "Files/raw/realtime/nasa_nrt")
]

print("\n📂 RAW REALTIME:")
for name, path in raw_dirs:
    total_p, files = list_and_count_files(path)
    print(f"\n   • {name} [{path}]: Total = {total_p:,} archivos Parquet.")
    if files:
        for f_path in files[:5]:
            print(f"       - {f_path}")
        if total_p > 5:
            print(f"       ... y {total_p - 5:,} archivos más.")

print("\n🛢️ BRONZE REALTIME:")
for t_name in ["bronze_dgt_traffic", "bronze_weather", "bronze_nasa_nrt"]:
    try:
        cnt = spark.table(t_name).count()
        print(f"   • {t_name:<22}: {cnt:,} registros Delta.")
    except Exception:
        print(f"   • {t_name:<22}: ❌ No encontrada en catálogo.")

# 5. AUDITORÍA DE TRAZABILIDAD
print("\n" + "="*80)
print("📊 AUDITORÍA DE DATOS Y MARCAS DE TRAZABILIDAD EN BRONZE")
print("="*80 + "\n")

def run_bronze_audit():
    tables = [t.name for t in spark.catalog.listTables() if t.name.startswith("bronze_")]
    meta_cols = ["landing_source_file", "ingestion_timestamp", "updated_source_file", "updated_timestamp"]

    for table_name in sorted(tables):
        print("-" * 80)
        print(f"🔎 AUDITANDO TABLA: {table_name.upper()}")
        print("-" * 80)
        try:
            df = spark.table(table_name)
            total_records = df.count()
            print(f"🔹 Nº Total de Registros: {total_records:,}")

            if total_records == 0:
                continue

            print("\n🛡️ AUDITORÍA DE METADATOS Y MARCAS TEMPORALES:")
            if "landing_source_file" in df.columns:
                dist_f = df.select(F.countDistinct("landing_source_file")).collect()[0][0]
                print(f"   • landing_source_file  [🟢 INGESTA INICIAL]: {dist_f:,} ficheros origen distintos.")

            if "ingestion_timestamp" in df.columns:
                ing_stats = df.select(F.min("ingestion_timestamp").alias("min_ing"), F.max("ingestion_timestamp").alias("max_ing")).collect()[0]
                print(f"   • ingestion_timestamp  [🟢 INGESTA INICIAL]: Desde [{ing_stats['min_ing']}] Hasta [{ing_stats['max_ing']}]")

            if "updated_source_file" in df.columns:
                upd_cnt = df.filter(F.col("updated_source_file").isNotNull()).count()
                print(f"   • updated_source_file  [🔴 ACTUALIZACIÓN]: {upd_cnt:,} registros modificados.")

            print("\n🔑 Completitud por Columna:")
            for col_name, dtype in df.dtypes:
                if col_name == "elements":
                    continue
                n_cnt = df.filter(F.col(col_name).isNull()).count()
                completeness = round(((total_records - n_cnt) / total_records) * 100, 2)
                print(f"   • {col_name:<30} ({dtype:<10}): Completitud: {completeness:>6.2f}%")

            print("\n📄 Muestra de Registros con Metadatos:")
            sample_cols = [c for c in meta_cols if c in df.columns]
            other_cols = [c for c in df.columns if c not in meta_cols and c != "elements"][:3]
            df.select(other_cols + sample_cols).show(3, truncate=40)
            print("\n")
        except Exception as e:
            print(f"❌ Error auditando la tabla {table_name}: {e}\n")

run_bronze_audit()